In [7]:
import torch

ckpt_check = torch.load('/kaggle/input/datasets/hibabou/poidsmecano/meccano_slowfast_mapped_clean.pth', 
                         map_location='cpu', weights_only=False)

if isinstance(ckpt_check, dict):
    print(f'Clés du dict : {list(ckpt_check.keys())}')
    inner = list(ckpt_check.values())[0]
    first_key = list(inner.keys())[0] if isinstance(inner, dict) else list(ckpt_check.keys())[0]
else:
    first_key = list(ckpt_check.keys())[0]

print(f'Première clé : {first_key}')
print(f'Total clés   : {len(ckpt_check) if not isinstance(list(ckpt_check.values())[0], dict) else len(list(ckpt_check.values())[0])}')

Clés du dict : ['model_state', 'arch', 'mapping_status', 'diagnostic']
Première clé : blocks.0.multipathway_blocks.0.conv.weight
Total clés   : 662


In [8]:
print(ckpt_check['arch'])
print(ckpt_check['mapping_status'])
print(ckpt_check['diagnostic'])

slowfast_r50_meccano
verified
{'total_params': 662, 'nan_params': 0, 'zero_params': 401}


In [11]:
import os
os.system('pip install fvcore -q')

import torch
import fvcore

slowfast_test = torch.hub.load('facebookresearch/pytorchvideo', 'slowfast_r50', pretrained=False)
msg = slowfast_test.load_state_dict(ckpt_check['model_state'], strict=False)
print(f'Missing : {msg.missing_keys}')

slowfast_test = slowfast_test.eval()

h = {}
handle = slowfast_test.blocks[5].register_forward_hook(lambda m, i, o: h.update({'out': o}))

with torch.no_grad():
    ts = torch.randn(1, 3, 8, 224, 224)
    tf = torch.randn(1, 3, 32, 224, 224)
    _ = slowfast_test([ts, tf])

handle.remove()
out = h['out']
print(f'Type : {type(out)}')
if isinstance(out, (list, tuple)):
    print(f'Nb éléments : {len(out)}')
    for i, o in enumerate(out):
        print(f'  out[{i}] shape : {o.shape}')
    dims = list(range(2, out[0].dim()))
    f = torch.cat([out[0].mean(dim=dims), out[1].mean(dim=dims)], dim=1)
else:
    print(f'Shape : {out.shape}')
    dims = list(range(2, out.dim()))
    f = out.mean(dim=dims) if len(dims) > 0 else out

print(f'Feature shape : {f.shape} | std={f.std().item():.4f}')

Using cache found in /root/.cache/torch/hub/facebookresearch_pytorchvideo_main


Missing : []
Type : <class 'torch.Tensor'>
Shape : torch.Size([1, 2304, 1, 1, 1])
Feature shape : torch.Size([1, 2304]) | std=0.8001


In [6]:
import os
print(os.listdir('/kaggle/input/datasets/hibabou/poidsmecano/'))

['meccano_slowfast_mapped_clean.pth']


In [13]:
import numpy as np
import os

feat_dir = '/kaggle/input/datasets/hibabou/featurestrain1/'
files = [f for f in os.listdir(feat_dir) if f.endswith('.npy')]
print(f'Total fichiers : {len(files)}')

stds = []
for f in files[:5]:  # vérifier les 5 premiers
    arr = np.load(os.path.join(feat_dir, f))
    std = arr.std()
    stds.append(std)
    print(f'  {f} | shape={arr.shape} | std={std:.4f}')

print(f'\nStd moyen : {np.mean(stds):.4f}')
print('✅ BON mapping (blocks[5])' if np.mean(stds) >= 0.35 else '❌ MAUVAIS mapping (blocks[2]) — re-extraire')

Total fichiers : 9
  train_p1_01_main_0_1_features.npy | shape=(1336, 2304) | std=0.4026
  train_p1_02_main_0_1_features.npy | shape=(2264, 2304) | std=0.3947
  train_p1_04_assy_2_1_features.npy | shape=(3096, 2304) | std=0.3832
  train_p1_04_assy_0_1_features.npy | shape=(4120, 2304) | std=0.4215
  train_p1_02_assy_1_2_features.npy | shape=(3632, 2304) | std=0.4049

Std moyen : 0.4014
✅ BON mapping (blocks[5])


Commancons extraction des autres 

In [9]:
import os, gc, shutil
import torch
import numpy as np

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLIP_LEN = 32
STRIDE   = 16
FEAT_DIR = '/kaggle/working/sf_features'
os.makedirs(FEAT_DIR, exist_ok=True)

BATCH = 'val_p2'  # changer selon notebook

SF_WEIGHTS = '/kaggle/input/datasets/hibabou/poidsmecano/meccano_slowfast_mapped_clean.pth'

BATCH_URLS = {
    'train_p1': 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/f58a5f18-9a71-4443-8a48-e59530df9ee8',
    'train_p2': 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/5aae5141-cece-41db-9681-47acd5300705',
    'train_p3': 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/4e2ed830-a1fe-40fa-9c5a-2aeae3964616',
    'train_p4': 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/3ff5409e-165f-4354-9fa4-a4fde639a7a6',
    'val_p1'  : 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/bb336949-248c-4ae6-82ef-107dbe61d10f',
    'val_p2'  : 'https://data.4tu.nl/file/b008dd74-020d-4ea4-a8ba-7bb60769d224/e899de9f-313a-482b-a6b5-e2de4f38412e',
}

print(f'DEVICE : {DEVICE}')
print(f'BATCH  : {BATCH}')

DEVICE : cuda
BATCH  : val_p2


chargement poids mappe

In [3]:
try:
    import fvcore
except:
    os.system('pip install fvcore -q')

slowfast = torch.hub.load('facebookresearch/pytorchvideo', 'slowfast_r50', pretrained=False)

# ← poids déjà mappés, pas besoin de convert_key()
ckpt = torch.load(SF_WEIGHTS, map_location='cpu', weights_only=False)
msg  = slowfast.load_state_dict(ckpt['model_state'], strict=False)

print(f'Arch    : {ckpt["arch"]}')
print(f'Status  : {ckpt["mapping_status"]}')
print(f'Missing : {len(msg.missing_keys)} (attendu 0)')

slowfast = slowfast.eval().to(DEVICE)
for p in slowfast.parameters():
    p.requires_grad = False

print('✅ SlowFast chargé')

Using cache found in /root/.cache/torch/hub/facebookresearch_pytorchvideo_main


Arch    : slowfast_r50_meccano
Status  : verified
Missing : 0 (attendu 0)
✅ SlowFast chargé


cellule extraction

In [10]:
from PIL import Image
import torchvision.transforms as T_img
from tqdm import tqdm

# ── Hooks ────────────────────────────────────────────────
feat_hook = {}

def hook_slow(module, input, output):
    feat_hook['slow'] = output

def hook_fast(module, input, output):
    feat_hook['fast'] = output

for m in slowfast.modules():
    m._forward_hooks.clear()

named = dict(slowfast.named_modules())
h1 = named['blocks.4.multipathway_blocks.0.res_blocks.2.activation'].register_forward_hook(hook_slow)
h2 = named['blocks.4.multipathway_blocks.1.res_blocks.2.activation'].register_forward_hook(hook_fast)
print('✅ Hooks ready (blocks.4 slow+fast)')

# ── Feature build ────────────────────────────────────────
def hook_to_feat(h):
    f0 = h['slow'].mean(dim=[3, 4])          # (1, 2048, T_slow)
    f1 = h['fast'].mean(dim=[3, 4])          # (1, 256,  T_fast)
    T  = f0.shape[2]
    f1 = torch.nn.functional.interpolate(f1, size=T, mode='linear', align_corners=False)
    feat = torch.cat([f0, f1], dim=1)        # (1, 2304, T)
    return feat.squeeze(0).permute(1, 0)     # (T, 2304)

# ── Transform ────────────────────────────────────────────
transform = T_img.Compose([
    T_img.Resize((256, 256)),
    T_img.CenterCrop(224),
    T_img.ToTensor(),
    T_img.Normalize(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225])
])

# ── Extract recording ────────────────────────────────────
def extract_recording(rgb_dir, out_path):
    if os.path.exists(out_path):
        return None
    frames = sorted([
        os.path.join(rgb_dir, f)
        for f in os.listdir(rgb_dir) if f.endswith('.jpg')
    ])
    if len(frames) < CLIP_LEN:
        return None
    feats = []
    with torch.no_grad():
        for s in range(0, len(frames) - CLIP_LEN + 1, STRIDE):
            feat_hook.clear()
            clip = []
            for fp in frames[s:s + CLIP_LEN]:
                try:
                    clip.append(transform(Image.open(fp).convert('RGB')))
                except:
                    clip.append(torch.zeros(3, 224, 224))
            c    = torch.stack(clip).permute(1, 0, 2, 3)
            slow = c[:, ::4].unsqueeze(0).to(DEVICE)
            fast = c.unsqueeze(0).to(DEVICE)
            _    = slowfast([slow, fast])
            feat = hook_to_feat(feat_hook)
            feats.append(feat.cpu().numpy())
            del slow, fast
            torch.cuda.empty_cache()
    if not feats:
        return None
    arr = np.vstack(feats).astype(np.float32)
    np.save(out_path, arr)
    del feats
    gc.collect()
    return arr.std()

# ── Download & extract ───────────────────────────────────
zip_path = f'/kaggle/working/{BATCH}.zip'
rgb_base = f'/kaggle/working/{BATCH}_rgb'

print(f'📥 Download {BATCH}...')
os.system(f'curl -L --retry 3 --retry-delay 5 -o {zip_path} "{BATCH_URLS[BATCH]}"')
os.makedirs(rgb_base, exist_ok=True)
os.system(f'unzip -q {zip_path} "*/rgb/*" -d {rgb_base}/')
os.system(f'rm {zip_path}')
print('Zip supprimé')
os.system('df -h /kaggle/working | tail -1')

# ── Scan recordings ──────────────────────────────────────
rgb_dirs = {}
for root, _, files in os.walk(rgb_base):
    if os.path.basename(root) == 'rgb' and files:
        rec = os.path.basename(os.path.dirname(root))
        rgb_dirs[rec] = root
print(f'Recordings : {len(rgb_dirs)}')

# ── Boucle extraction ────────────────────────────────────
stds = []
for rec, rgb_dir in tqdm(rgb_dirs.items(), desc=f'Extract {BATCH}'):
    out = os.path.join(FEAT_DIR, f'{BATCH}_{rec}_features.npy')
    std = extract_recording(rgb_dir, out)
    if std is not None:
        stds.append(std)
        print(f'  {rec} : std={std:.3f}')

# ── Cleanup ──────────────────────────────────────────────
h1.remove()
h2.remove()
shutil.rmtree(rgb_base)
gc.collect()
torch.cuda.empty_cache()

done = [f for f in os.listdir(FEAT_DIR) if f.endswith('.npy')]
print(f'\nTotal : {len(done)} features sauvegardées')
if stds:
    print(f'Std moyen : {np.mean(stds):.4f}')
    print('✅ OK' if np.mean(stds) >= 0.35 else '❌ Std faible')
os.system('df -h /kaggle/working | tail -1')

✅ Hooks ready (blocks.4 slow+fast)
📥 Download val_p2...


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5934M  100 5934M    0     0  24.5M      0  0:04:01  0:04:01 --:--:-- 29.4M


Zip supprimé
/dev/loop1       20G  2.8G   17G  15% /kaggle/working
Recordings : 9


Extract val_p2:  11%|█         | 1/9 [01:17<10:21, 77.65s/it]

  26_assy_0_1 : std=0.384


...

  26_assy_1_5 : std=0.388


Extract val_p2: 100%|██████████| 9/9 [09:43<00:00, 64.85s/it]

  20_assy_3_6 : std=0.421



Total : 16 features sauvegardées
Std moyen : 0.3925
✅ OK
/dev/loop1       20G  208M   20G   2% /kaggle/working


0

In [ ]:
Cellule 4 — Diagnostic std + cosine similarity

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('.npy')])
print(f'Total fichiers : {len(files)}\n')

all_stds, all_cos = [], []

for f in files:
    arr = np.load(os.path.join(FEAT_DIR, f))
    std = arr.std()
    cos = cosine_similarity(arr[:-1], arr[1:]).diagonal().mean()
    all_stds.append(std)
    all_cos.append(cos)
    print(f'  {f} | shape={arr.shape} | std={std:.4f} | cos={cos:.4f}')

print(f'\n── GLOBAL ──────────────────────')
print(f'Std moyen     : {np.mean(all_stds):.4f}')
print(f'Cos sim moyen : {np.mean(all_cos):.4f}')
print('✅ Features correctes' if np.mean(all_stds) >= 0.35 else '❌ Re-extraire')

Total fichiers : 16

  val_p1_05_assy_0_1_features.npy | shape=(1448, 2304) | std=0.4056 | cos=0.6164
  val_p1_05_assy_2_2_features.npy | shape=(1152, 2304) | std=0.4229 | cos=0.6046
  val_p1_05_main_0_1_features.npy | shape=(680, 2304) | std=0.4207 | cos=0.6056
  val_p1_14_assy_0_1_features.npy | shape=(1488, 2304) | std=0.4221 | cos=0.6143
  val_p1_14_main_0_1_features.npy | shape=(832, 2304) | std=0.4118 | cos=0.6513
  val_p1_14_main_2_2_features.npy | shape=(688, 2304) | std=0.4043 | cos=0.6553
  val_p1_14_main_2_3_features.npy | shape=(824, 2304) | std=0.4061 | cos=0.6252
  val_p2_20_assy_0_1_features.npy | shape=(1416, 2304) | std=0.3911 | cos=0.6211
  val_p2_20_assy_3_6_features.npy | shape=(1472, 2304) | std=0.4207 | cos=0.5786
  val_p2_20_main_0_1_features.npy | shape=(1024, 2304) | std=0.3852 | cos=0.5880
  val_p2_24_assy_0_1_features.npy | shape=(1064, 2304) | std=0.3911 | cos=0.5866
  val_p2_24_assy_2_4_features.npy | shape=(1464, 2304) | std=0.4224 | cos=0.5979
  val_p2_24

In [13]:
import os



# ── val_p2 ───────────────────────────────────────────────
zip_val2 = '/kaggle/working/sf_features_val_p2_only.zip'
files_val2 = [f for f in os.listdir('/kaggle/working/sf_features/') if f.startswith('val_p2')]
files_str2 = ' '.join([f'/kaggle/working/sf_features/{f}' for f in files_val2])
os.system(f'zip {zip_val2} {files_str2}')
print(f'val_p2 : {len(files_val2)} fichiers')
os.system(f'ls -lh {zip_val2}')

  adding: kaggle/working/sf_features/val_p2_24_assy_0_1_features.npy (deflated 33%)
  adding: kaggle/working/sf_features/val_p2_20_main_0_1_features.npy (deflated 33%)
  adding: kaggle/working/sf_features/val_p2_20_assy_0_1_features.npy (deflated 31%)
  adding: kaggle/working/sf_features/val_p2_24_main_0_1_features.npy (deflated 33%)
  adding: kaggle/working/sf_features/val_p2_24_assy_2_4_features.npy (deflated 31%)
  adding: kaggle/working/sf_features/val_p2_20_assy_3_6_features.npy (deflated 30%)
  adding: kaggle/working/sf_features/val_p2_26_main_0_1_features.npy (deflated 33%)
  adding: kaggle/working/sf_features/val_p2_26_assy_1_5_features.npy (deflated 34%)
  adding: kaggle/working/sf_features/val_p2_26_assy_0_1_features.npy (deflated 31%)
val_p2 : 9 fichiers
-rw-r--r-- 1 root root 71M May 13 19:44 /kaggle/working/sf_features_val_p2_only.zip


0

si val continet meme classe que train noanalisation lissage tester version compressed